# Statlog German Credit — Feature Engineering Deep Dive

In [ ]:

from pathlib import Path
import sys
ROOT = Path.cwd()
if ROOT.name.endswith('_exp'):
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'hyperack_exp'))
from general_pipeline.playbook.ladder import load_raw_xy
from general_pipeline.playbook.features import build_feature_matrix
from shared.protocol import evaluate
from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import pandas as pd

KEY = 'german_credit'
model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', LGBMClassifier(random_state=42, verbosity=-1)),
])
rows = []
for mode in ['safe', 'unsafe']:
    Xtr, ytr, Xte, yte, meta = load_raw_xy(KEY, mode)
    for stage in ['raw', 'logs', 'ratios', 'interactions', 'full_fe', 'selected']:
        A, B, fe = build_feature_matrix(Xtr, ytr, Xte, stage=stage)
        m = evaluate(model, A, ytr, B, yte)
        rows.append({'mode': mode, 'stage': stage, 'feats': A.shape[1], 'roc_auc': m['roc_auc'], 'f1': m['f1']})
pd.DataFrame(rows).sort_values(['mode', 'roc_auc'], ascending=[True, False])
